# Practice: Your First Hour With an Unfamiliar Corpus

**Computational Text Analysis Pack · Module 00 — Text as Data**

*Technique:* A reproducible first-look protocol before any analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/00-foundations/00b_practice_first_look.ipynb)

Data: `masakhanews.csv`, `colonial_texts.csv`, `african_legal.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'colonial_texts.csv', 'african_legal.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_colonial = pd.read_csv('colonial_texts.csv')
df_legal = pd.read_csv('african_legal.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## The question

> **Someone hands you a corpus. What do you do in the first hour, before you
> have any hypothesis at all?**

There is a protocol, and running it saves weeks. Most of the mistakes this
course spends 28 modules correcting are visible in the first hour if you look.

### How to use this notebook

A worked case, not a tutorial. Run it once for the shape of the argument, then
change the question at the top to one of your own.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import numpy as np
CORPORA = {"masakhanews": df, "colonial_texts": df_colonial, "african_legal": df_legal}

print("STEP 1 — shape. What is a row, and how many are there?\n")
for name, frame in CORPORA.items():
    print(f"{name:<18}{len(frame):>8,} rows  {frame.shape[1]:>3} columns")
    print(f"{'':18}columns: {list(frame.columns)}")
    print()

In [ ]:
print("STEP 2 — the text column. How long, how variable?\n")
for name, frame in CORPORA.items():
    col = "text"
    lens = frame[col].astype(str).str.split().str.len()
    print(f"{name:<18}median {lens.median():>7,.0f} words   "
          f"range {lens.min():>6,} - {lens.max():>7,}   "
          f"empty/short: {(lens < 10).sum():>5,}")
print()
print("A big range means the unit is inconsistent — some rows are pages, some")
print("are books. That decides whether you can compare rows at all.")

In [ ]:
print("STEP 3 — duplicates and boilerplate. Both silently distort every count.\n")
for name, frame in CORPORA.items():
    txt = frame["text"].astype(str)
    exact = len(txt) - txt.nunique()
    heads = Counter(" ".join(t.split()[:8]) for t in txt)
    top_head, n_head = heads.most_common(1)[0]
    print(f"{name:<18}exact duplicate rows: {exact:>5,}")
    print(f"{'':18}most repeated opening ({n_head}x): {top_head[:56]!r}")
    print()

In [ ]:
print("STEP 4 — the metadata skew. Who or what dominates?\n")
for name, frame in CORPORA.items():
    for col in ("author", "language", "category"):
        if col in frame.columns and frame[col].notna().any():
            vc = frame[col].value_counts()
            share = vc.iloc[0] / len(frame)
            print(f"{name:<18}{col:<10}{len(vc):>4} values; "
                  f"largest '{str(vc.index[0])[:24]}' holds {share:.0%}")
    print()
print("Anything holding more than a third is a finding waiting to happen — every")
print("aggregate you compute will substantially be about it.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (name, frame) in zip(axes, CORPORA.items()):
    lens = frame["text"].astype(str).str.split().str.len()
    ax.hist(np.log10(lens.clip(lower=1)), bins=30, color="steelblue")
    ax.set_title(name, fontsize=9)
    ax.set_xlabel("log10 words per row")
plt.tight_layout()
print("Multiple peaks mean multiple document types mixed in one file. That is")
print("worth knowing before you average anything across them.")

## The first-hour checklist

1. **What is a row?** Article, passage, page, book. Everything downstream assumes it.
2. **How variable is length?** A wide range means rows are not comparable units.
3. **Are there duplicates or shared boilerplate?** Both inflate every frequency.
4. **What dominates the metadata?** One author, one language, one outlet.
5. **What is missing?** Not just null cells — which authors, languages, periods
   were never collected. Module 22c makes this a formal audit.

Write the answers down. They become your methods section, and they tell you
which of the next 28 modules your corpus can actually support.

### 🔧 Your turn 1

Run all five steps on a corpus of your own before doing anything else with it. Which step surprised you?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Step 3 surprises people most often — near-duplicate rows and shared boilerplate
are extremely common in scraped or archival data, and they inflate exactly the
frequencies that look most interesting. Finding them in the first hour is cheap;
finding them after you have written the results section is expensive, and after
publication it is a correction.

</details>